##Chapter 6: Tools for Model Training and Experimenting/Project:
* Retrieval Augmented Generation Chatbot/CH6-01-Assembling ChatBot.py

In [0]:
!pip install -q mlflow>=2.17 langchain>=0.3 langchain-community>=0.3 databricks-vectorsearch databricks-sdk

In [0]:
from mlflow.deployments import get_deploy_client
deploy_client = get_deploy_client("databricks")
endpoints = deploy_client.list_endpoints()
for endpoint in endpoints:
    print(endpoint['name'])

In [0]:
import os 
# url used to send the request to your model from the serverless endpoint
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
db_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get() 
#db_token = dbutils.secrets.get("mlaction", "rag_sp_token")
#db_host = dbutils.secrets.get("mlaction", "rag_sp_host")
os.environ['DATABRICKS_TOKEN'] = db_token
os.environ['DATABRICKS_HOST'] = host

In [0]:
from databricks.vector_search.client import VectorSearchClient
from langchain_community.vectorstores import DatabricksVectorSearch
from langchain_community.embeddings import DatabricksEmbeddings

catalog = "porya_catalog"
database_name = "default"

# NOTE: your question embedding model must match the one used in the chunk in the previous model 
embedding_model = DatabricksEmbeddings(endpoint="databricks-gte-large-en")
vsc_endpoint_name = "one-env-shared-endpoint-1" #"ml_action_vs"
index_name = f"{catalog}.{database_name}.docs_vsc_idx_cont"

def get_retriever(persist_dir: str = None):
    #Get the vector search index
    vsc = VectorSearchClient(workspace_url=os.environ['DATABRICKS_HOST'], personal_access_token=os.environ["DATABRICKS_TOKEN"])
    print("\n")
    vs_index = vsc.get_index(
        endpoint_name=vsc_endpoint_name,
        index_name=index_name)
    # Create the retriever
    vectorstore = DatabricksVectorSearch(
        vs_index, text_column="content", embedding=embedding_model)
    return vectorstore.as_retriever()

# test our retriever
vectorstore = get_retriever()
similar_documents = vectorstore.invoke("What is GPT? ")
print(f"Relevant documents: {similar_documents[0]}")

In [0]:
import langchain
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_models import ChatDatabricks

# Using Foundational Model from Databricks
chat_model = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", 
                            max_tokens = 200)

TEMPLATE = """
You are an assistant for the AI Swat Team. You are answering questions related to the GenerativeAI and LLM's and how they impact humans life, labour, economic and financial impact. If the question is not related to one of these topics, kindly decline to answer. If you don't know the answer, just say that you don't know, don't try to make up an answer. Keep the answer as concise as possible. Use the following pieces of context to answer the question at the end:
{context}
Question: {question}
Answer:
"""
prompt = PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = get_retriever()

# Modern LCEL chain (replaces legacy RetrievalQA)
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

# COMMAND ----------

langchain.debug = False
question = "Will AI impact work forces in the US?"
answer = chain.invoke(question)
print(answer, "\n")

question = "Can LLM's impact wages and how?"
answer = chain.invoke(question)
print(answer, "\n")

In [0]:
import mlflow
from mlflow.models import infer_signature
from mlflow.tracking.client import MlflowClient
import os

# Replacement for mlia_utils helpers
current_user = spark.sql("SELECT current_user()").first()[0]

def get_latest_model_version(model_name):
    client = MlflowClient()
    versions = client.search_model_versions(f"name='{model_name}'")
    return max([int(v.version) for v in versions])

# create experiment if does not exist 
experiment_path = f"/Users/{current_user}/mlaction_chatbot_rag"
mlflow.set_experiment(experiment_path) 

# Write chain definition to file (models-from-code pattern)
chain_code = '''
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_models import ChatDatabricks
from langchain_community.vectorstores import DatabricksVectorSearch
from langchain_community.embeddings import DatabricksEmbeddings
from databricks.vector_search.client import VectorSearchClient
import mlflow

catalog = "porya_catalog"
database_name = "default"

embedding_model = DatabricksEmbeddings(endpoint="databricks-gte-large-en")
vsc_endpoint_name = "one-env-shared-endpoint-1"
index_name = f"{catalog}.{database_name}.docs_vsc_idx_cont"

def get_retriever(persist_dir=None):
    vsc = VectorSearchClient(
        workspace_url=os.environ.get("DATABRICKS_HOST", ""),
        personal_access_token=os.environ.get("DATABRICKS_TOKEN", "")
    )
    vs_index = vsc.get_index(endpoint_name=vsc_endpoint_name, index_name=index_name)
    vectorstore = DatabricksVectorSearch(vs_index, text_column="content", embedding=embedding_model)
    return vectorstore.as_retriever()

chat_model = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=200)

TEMPLATE = """
You are an assistant for the AI Swat Team. You are answering questions related to the GenerativeAI and LLM\'s and how they impact humans life, labour, economic and financial impact. If the question is not related to one of these topics, kindly decline to answer. If you don\'t know the answer, just say that you don\'t know, don\'t try to make up an answer. Keep the answer as concise as possible. Use the following pieces of context to answer the question at the end:
{context}
Question: {question}
Answer:
"""
prompt = PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])

def format_docs(docs):
    return "\\n\\n".join(doc.page_content for doc in docs)

retriever = get_retriever()

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

mlflow.models.set_model(chain)
'''

chain_path = "chain.py"
with open(chain_path, "w") as f:
    f.write(chain_code)

print(f"Chain code written to {chain_path}")

# COMMAND ----------

mlflow.set_registry_uri("databricks-uc")
model_name = f"{catalog}.{database_name}.mlaction_chatbot_model"

with mlflow.start_run(run_name="mlaction_chatbot_rag") as run:
    signature = infer_signature(question, answer)
    model_info = mlflow.langchain.log_model(
        lc_model=chain_path,
        artifact_path="chain",
        registered_model_name=model_name,
        pip_requirements=[
            "mlflow>=2.17",
            "langchain>=0.3",
            "langchain-community>=0.3",
            "databricks-vectorsearch",
        ],
        input_example=question,
        signature=signature
    )

print(f"Model registered: {model_name} version {get_latest_model_version(model_name)}")